# liar_report

> Phase 15 analysis (`TRACE_PLAN`): `mechanics` (the measures M1-M6 of one traced run with liars), `lossAccount` (the honest loss of one board split into eight terms that sum to it exactly), `traceSummaries` (per runner and N, the changes from N 8 to 32, Haiku minus numpy, the hypotheses), `writeTrace` and `figTrace`. All findings are exploratory; the variants are retrodictions.

In [ ]:
#| default_exp liar_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Measures and accounting

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.liar_grid import _boot, _pairs, nChange, numpyChanges
from explRL.llm_figures import _save
from explRL.llm_report import NAN, _stat

MEASURES = ('falseVisible', 'contaminated', 'wrongContam', 'wrongClean', 'suppressed', 'relayPerLie', 'relayAny',
            'honestErrors', 'liarHeld', 'solvedLiarHeld', 'solvedOther', 'recoverRounds', 'contamUnsolved')
TERMS = tuple('abcdefgh')


def _r(num, den):
    return float(num) / den if den else NAN


def _contam(trace):
    "{(t, agent): contaminated?} from a trace."
    return {(e.t, e.i): e.contam for e in trace}


def mechanics(trace, board, nLiar):
    "M1-M6 (TRACE_PLAN) of one traced run; honest agents are nLiar .. N - 1."
    ct, sl = _contam(trace), board.shareLog
    subs = [s for s in board.submits if s[1] >= nLiar]
    con = [ct.get((s[0], s[1]), False) for s in subs]
    wrong = [s[4] == 'wrong' for s in subs]
    first, liarSeen, relays, errors, honest, relayed = {}, set(), 0, 0, 0, set()
    for s in sl:
        first.setdefault((s[2], s[3]), s)
        if s[1] < nLiar:
            liarSeen.add((s[2], s[3], s[4]))
            continue
        honest += 1
        if s[6] and (s[2], s[3], s[4]) in liarSeen:
            relays += 1
            relayed.add((s[2], s[3], s[4]))
        elif s[6]:
            errors += 1
    lies = [s for s in sl if s[1] < nLiar]
    liarFirst = [x for x, s in first.items() if s[1] < nLiar]
    trueLater = {(s[2], s[3]) for s in sl if s[1] >= nLiar and not s[6]}
    held = {k: any(h < nLiar for hs in s.holders for h in hs) for k, s in board.tasks.items()}
    solved = {k: s.solvedBy >= 0 for k, s in board.tasks.items()}
    tSolve = {k: t for t, _, k, _ in board.solveLog}
    rec = [tSolve[s[2]] - s[0] for s, w in zip(subs, wrong) if w and s[2] in tSolve and tSolve[s[2]] >= s[0]]
    cks = {s[2] for s, c in zip(subs, con) if c}
    return dict(
        falseVisible=float(np.mean([e.falseVisible / e.nOpen for e in trace if e.nOpen])) if trace else NAN,
        contaminated=_r(sum(con), len(subs)),
        wrongContam=_r(sum(w for w, c in zip(wrong, con) if c), sum(con)),
        wrongClean=_r(sum(w for w, c in zip(wrong, con) if not c), len(subs) - sum(con)),
        suppressed=_r(sum(x not in trueLater for x in liarFirst), len(liarFirst)),
        relayPerLie=_r(relays, len(lies)), relayAny=_r(sum((s[2], s[3], s[4]) in relayed for s in lies), len(lies)),
        honestErrors=_r(errors, honest), liarHeld=_r(sum(held.values()), len(held)),
        solvedLiarHeld=_r(sum(solved[k] for k in held if held[k]), sum(held.values())),
        solvedOther=_r(sum(solved[k] for k in held if not held[k]), len(held) - sum(held.values())),
        recoverRounds=float(np.mean(rec)) if rec else NAN, contamUnsolved=_r(sum(not solved[k] for k in cks), len(cks)))


def lossAccount(clean, liar):
    """The honest loss of one board (TraceRuns of the clean and the liar cell, same N and seed) split into the terms
    a-h of TRACE_PLAN, each per honest agent-turn; 'total' is their sum, 'loss' welfareHonest clean - liar."""
    bc, bl, nL = clean.board, liar.board, liar.nLiar
    dC, dL, pen = (bc.nAgents - clean.nLiar) * clean.T, (bl.nAgents - nL) * liar.T, bl.penalty
    sC = {k: s.value for k, s in bc.tasks.items() if s.solvedBy >= clean.nLiar}
    sL = {k: s.value for k, s in bl.tasks.items() if s.solvedBy >= nL}
    ct = _contam(liar.trace)
    wl = [s for s in bl.submits if s[1] >= nL and s[4] == 'wrong']
    wc = sum(1 for s in bc.submits if s[1] >= clean.nLiar and s[4] == 'wrong')
    cw = sum(ct.get((s[0], s[1]), False) for s in wl)
    T = dict.fromkeys(TERMS, 0.0)
    T['a'], T['b'] = pen * cw / dL, pen * (len(wl) - cw) / dL - pen * wc / dC
    falsePost = {s[2] for s in bl.shareLog if s[6]}
    for k, v in sC.items():
        if k in sL:
            T['h'] += v * (1 / dC - 1 / dL)
            continue
        sl = bl.tasks.get(k)                                       # None: the liar cell never reached subtask k
        t = ('e' if sl is not None and 0 <= sl.solvedBy < nL else 'c' if k in falsePost else
             'd' if any(h < nL for hs in bc.tasks[k].holders for h in hs) else 'f')
        T[t] += v / dC
    T['g'] = -sum(v / dL for k, v in sL.items() if k not in sC)
    return dict(T, total=sum(T.values()), loss=clean.rows[0]['welfareHonest'] - liar.rows[0]['welfareHonest'])

## The summaries

In [ ]:
#| export
def traceSummaries(haiku, numpy, variants=None):
    """TRACE_PLAN: per runner and N the measures, the terms and the honest loss (mean, CI, per seed); their changes
    N 8 -> 32 (and, secondary, N 16 and 32 pooled against N 8); Haiku's change minus numpy's; H15-1 to H15-4. haiku, numpy: {key: TraceRun}; variants: {name: rows}."""
    S = dict(perN={}, change={}, pooled={}, diff={}, variants={}, hyp={})
    vals = {}
    for name, runs in (('haiku', haiku), ('numpy', numpy)):
        vals[name] = {}
        for N, seeds in sorted(_pairs(runs).items()):
            recs = [dict(**mechanics(d['liar'].trace, d['liar'].board, d['liar'].nLiar), **lossAccount(d['clean'], d['liar']),
                         pooledShare=d['clean'].rows[0]['pooledShare']) for _, d in sorted(seeds.items()) if len(d) == 2]
            vals[name][N] = {m: [x[m] for x in recs] for m in (recs[0] if recs else {})}
        S['perN'][name] = {N: {m: _stat(v) for m, v in d.items()} for N, d in vals[name].items()}
        S['change'][name] = {m: nChange(vals[name][8][m], vals[name][32][m]) for m in vals[name].get(8, {})}
        S['pooled'][name] = {m: nChange(vals[name][8][m], [x for N in (16, 32) for x in vals[name].get(N, {}).get(m, [])])
                             for m in vals[name].get(8, {})}           # secondary: N 16 and 32 pooled against N 8
    h, n = vals['haiku'], vals['numpy']
    S['diff'] = {m: _boot(lambda a, b, c, d: (b.mean() - a.mean()) - (d.mean() - c.mean()), (h[8][m], h[32][m], n[8][m], n[32][m]))
                 for m in S['change']['haiku'] if m in S['change']['numpy']}
    ac = lambda N: np.array(h[N]['a']) + np.array(h[N]['c'])
    ratio = _boot(lambda a8, a32, t8, t32: (a32.mean() - a8.mean()) / (t32.mean() - t8.mean()),
                  (ac(8), ac(32), h[8]['total'], h[32]['total']))
    hc, nc = S['change']['haiku'], S['change']['numpy']
    for name, rows in (variants or {}).items():
        S['variants'][name] = numpyChanges(rows)
    S['hyp'] = dict(
        H15_1=dict(haiku=hc['contaminated']['sign'], numpy=nc['contaminated']['sign'],
                   supported=hc['contaminated']['sign'] in '+0' and nc['contaminated']['sign'] == '-'),
        H15_2=dict(perN={N: S['perN']['haiku'][N]['relayPerLie']['mean'] for N in h}, sign=hc['relayPerLie']['sign'],
                   supported=all(S['perN']['haiku'][N]['relayPerLie']['mean'] > 0 for N in h) and hc['relayPerLie']['sign'] in '+0'),
        H15_3=dict(ratio, supported=ratio['mean'] > 0.5),
        H15_4=dict(haiku=hc['loss']['sign'], variants={v: x['honestLoss']['change']['sign'] for v, x in S['variants'].items()},
                   supported=any(x['honestLoss']['change']['sign'] == hc['loss']['sign'] for x in S['variants'].values())))
    return S

## Report and figure

In [ ]:
#| export
def _g(x, d=2):
    return 'n/a' if x is None or x != x else f'{x:.{d}f}'


def _ci(v, d=2):
    return f"{_g(v['mean'], d)} [{_g(v['ci'][0], d)}, {_g(v['ci'][1], d)}]"


def writeTrace(path, S, meta):
    "report.md: provenance, the hypotheses, measures and terms per runner and N, changes, Haiku minus numpy, variants."
    L = [f"# Phase 15: the mechanism behind liar damage at scale ({meta['mode']})", '', 'Exploratory, pre-specified in '
         '`TRACE_PLAN`; the variants are retrodictions.', '', f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; "
         f"Haiku cells ({meta['haikuSource']}): {meta['haikuCells']}, numpy cells: {meta['numpyCells']}; new API spend $0", '',
         '## Hypotheses', '']
    for k, v in S['hyp'].items():
        L.append(f"- **{k.replace('_', '-')}**: {'supported' if v['supported'] else 'not supported'} "
                 f"({', '.join(f'{a}: {b}' for a, b in v.items() if a != 'supported')})")
    for name in ('haiku', 'numpy'):
        P = S['perN'][name]
        cols = [m for m in MEASURES + TERMS + ('total', 'loss', 'pooledShare') if any(m in d for d in P.values())]
        L += ['', f'## {name}: per N (mean [95% CI] over seeds)', '', '| measure | ' + ' | '.join(f'N {N}' for N in P) +
              ' | change N 8 -> 32 | N 16 + 32 vs 8 |', '|---|' + '---|' * (len(P) + 2)]
        L += [f"| {m} | " + ' | '.join(_ci(P[N][m], 3) for N in P) + f" | {_ci(S['change'][name][m], 3)} "
              f"{S['change'][name][m]['sign']} | {_ci(S['pooled'][name][m], 3)} {S['pooled'][name][m]['sign']} |"
              for m in cols if m in S['change'][name]]
    L += ['', "## Haiku's change minus numpy's (N 8 -> 32)", '', '| measure | difference [95% CI] |', '|---|---|']
    L += [f'| {m} | {_ci(v, 3)} |' for m, v in S['diff'].items()]
    if S['variants']:
        L += ['', '## Numpy variants (retrodiction; N 8 -> 32)', '', '| variant | honest loss | pooled share |', '|---|---|---|']
        L += [f"| {v} | {_ci(x['honestLoss']['change'], 3)} {x['honestLoss']['change']['sign']} | "
              f"{_ci(x['pooledShare']['change'], 3)} {x['pooledShare']['change']['sign']} |" for v, x in S['variants'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def figTrace(out, S):
    "liar.png: contaminated share and false relays per lie vs N (Haiku, numpy), and the terms' changes N 8 -> 32."
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
    for ax, m in zip(axs[:2], ('contaminated', 'relayPerLie')):
        for name, mk in (('haiku', 'o-'), ('numpy', 's--')):
            P = S['perN'][name]
            ax.plot(list(P), [P[N][m]['mean'] for N in P], mk, label=name)
        ax.set(xscale='log', xlabel='team size N', ylabel=m)
        ax.legend(fontsize=8)
    x = np.arange(len(TERMS))
    for off, name in ((-0.2, 'haiku'), (0.2, 'numpy')):
        axs[2].bar(x + off, [S['change'][name][t]['mean'] for t in TERMS], 0.4, label=name)
    axs[2].set_xticks(x, TERMS)
    axs[2].axhline(0, color='k', lw=0.5)
    axs[2].set(ylabel='change N 8 -> 32 (per agent-turn)', title='honest-loss terms')
    axs[2].legend(fontsize=8)
    return _save(fig, out / 'liar.png')

## Tests

In [ ]:
import json
import numpy as np
from explRL.liar_report import *
from explRL.liar_grid import nChange
from explRL.liar_trace import traceCells
from explRL.liar_grid import forecastCells

# nChange is tools/scale_forecast.py's change
import importlib.util, pathlib
root = pathlib.Path('..') if pathlib.Path('../tools').exists() else pathlib.Path('.')
spec = importlib.util.spec_from_file_location('sf', root / 'tools/scale_forecast.py')
sf = importlib.util.module_from_spec(spec); spec.loader.exec_module(sf)
rng = np.random.default_rng(3)
for _ in range(5):
    lo, hi = rng.normal(0, 1, 5), rng.normal(0.5, 1, 7)
    assert nChange(lo, hi) == sf.change(lo, hi)

In [ ]:
# AC-3: on real numpy pairs the eight terms sum to the honest loss to 1e-12
runs = traceCells(forecastCells(sizes=(8, 32), seeds=range(2000, 2004)))
from explRL.liar_grid import _pairs
P = _pairs(runs)
for N, seeds in P.items():
    for sd, d in seeds.items():
        A = lossAccount(d['clean'], d['liar'])
        assert abs(A['total'] - A['loss']) < 1e-12, (N, sd, A)
        assert A['h'] <= 0 and A['g'] <= 0 and A['a'] >= 0          # per-capita and liar-only solves enter negatively
        m = mechanics(d['liar'].trace, d['liar'].board, d['liar'].nLiar)
        assert m['relayPerLie'] == 0 and not m['honestErrors'] > 0   # rule agents share only clues they hold (nan: no shares)
        assert 0 <= m['contaminated'] <= 1 and 0 < m['liarHeld'] < 1 and 0 <= m['suppressed'] <= 1

In [ ]:
# AC-3: the measures and the accounting on a hand-made run (two agents, agent 0 a liar)
from types import SimpleNamespace as NS
from explRL.liar_trace import Entry, TraceRun
task = lambda v, holders, by=-1: NS(value=v, holders=holders, solvedBy=by)
lb = NS(nAgents=3, penalty=0.5,
        shareLog=[(0., 0, 1, 0, 5, False, True), (1., 1, 1, 0, 5, False, True), (2., 2, 1, 0, 3, False, False),
                  (3., 0, 2, 1, 4, False, True), (3., 1, 3, 0, 1, False, True)],
        submits=[(1., 1, 1, None, 'wrong'), (2., 2, 2, None, 'wrong'), (4., 1, 1, None, 'solved'), (5., 2, 3, None, 'solved')],
        solveLog=[(4., 1, 1, True), (5., 2, 3, False)],
        tasks={1: task(2, [frozenset({0}), frozenset({2})], 1), 2: task(1, [frozenset({1}), frozenset({2})]),
               3: task(3, [frozenset({1}), frozenset({2})], 2), 4: task(1, [frozenset({0}), frozenset({1})], 0)})
tr = [Entry(1., 1, 1, {0: (5, False)}, True, 1, 4), Entry(2., 2, 2, {}, False, 1, 4),
      Entry(4., 1, 1, {0: (3, True)}, False, 0, 3), Entry(5., 2, 3, {0: (1, True)}, False, 0, 2)]
m = mechanics(tr, lb, 1)
assert m['contaminated'] == 0.25 and m['wrongContam'] == 1.0 and m['wrongClean'] == 1 / 3
assert m['relayPerLie'] == 0.5 and m['relayAny'] == 0.5 and m['honestErrors'] == 1 / 3    # agent 1 relays (1, 0, 5); (3, 0, 1) is an error
lb2 = NS(**{**vars(lb), 'shareLog': [(0., 1, 3, 0, 1, False, True), (1., 0, 3, 0, 1, False, True)]})
m2 = mechanics(tr, lb2, 1)
assert m2['relayAny'] == 0.0 and m2['relayPerLie'] == 0.0 and m2['honestErrors'] == 1.0   # the error came first: no relay
assert m['suppressed'] == 0.5          # slots first posted by the liar: (1, 0) (a true post followed) and (2, 1) (none)
assert m['liarHeld'] == 0.5 and m['solvedLiarHeld'] == 1.0 and m['solvedOther'] == 0.5
assert m['recoverRounds'] == 3.0 and m['contamUnsolved'] == 0.0
assert abs(m['falseVisible'] - (1 / 4 + 1 / 4) / 4) < 1e-12
# accounting by hand: clean board (no liars), 3 honest agents, T 10: solved 1 (2), 2 (1), 4 (1); one wrong answer
cb = NS(nAgents=3, penalty=0.5, shareLog=[], submits=[(3., 0, 2, None, 'wrong')], solveLog=[],
        tasks={1: task(2, lb.tasks[1].holders, 0), 2: task(1, lb.tasks[2].holders, 1), 3: task(3, lb.tasks[3].holders),
               4: task(1, lb.tasks[4].holders, 2), 5: task(2, [frozenset({1}), frozenset({2})], 1)})
wC, wL = (2 + 1 + 1 + 2 - 0.5) / 30, (2 + 3 - 0.5 * 2) / 20
A = lossAccount(TraceRun([dict(welfareHonest=wC)], [], [], cb, 0, 10), TraceRun([dict(welfareHonest=wL)], [], tr, lb, 1, 10))
expect = dict(a=0.5 / 20, b=0.5 / 20 - 0.5 / 30, c=1 / 30, d=0.0, e=1 / 30, f=2 / 30, g=-3 / 20, h=2 * (1 / 30 - 1 / 20))
assert all(abs(A[k] - v) < 1e-12 for k, v in expect.items()), A
assert abs(A['total'] - A['loss']) < 1e-12

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()